# 11 · Primer modelo: regresión logística

Queremos saber si las condiciones anteriores del mercado ayudan a distinguir qué señales superarán al efectivo. Entrenaremos un filtro para momentum y otro para reversión, conservando las diez características.

**Recorrido previo:** 01 → 02 → 07 → 08 → 10. El 09 no es una entrada: sus asociaciones no seleccionan nuestras variables. Selecciona el entorno `.venv`, el extra `models` incorpora scikit-learn: `python -m pip install -e ".[models]"`.

Aquí solo se utiliza **1927–1989**. CPCV permite bloques de entrenamiento posteriores a validación, sus resultados son de descubrimiento y no representan una inversión en tiempo real. Las medias mensuales de Shiller y los dividendos reconstruidos conservan las limitaciones ya explicadas. El [protocolo](../docs/primer_modelo.md) fija las decisiones antes del primer ajuste.


## 1. Qué aprende la logística

La etiqueta vale uno si seguir la señal durante el mes objetivo, con los costes del escenario central, habría superado al efectivo. Cero indica que no lo habría superado. No es directamente una etiqueta de subida o bajada del índice, en una venta corta puede ser favorable una caída. Los meses sin señal no son ejemplos negativos.

La logística combina las diez características transformadas y convierte su suma en una probabilidad entre cero y uno. Por ejemplo, 0,60 significa que el modelo estima un 60 % de probabilidad de etiqueta uno bajo los datos y supuestos del estudio, sin garantizar que esa estimación sea correcta.

Fijamos penalización **L2**, `C=1`, intercepto y optimizador `lbfgs`, sin ponderación especial de clases, selección de variables o búsqueda de parámetros. L2 reduce coeficientes grandes para limitar el sobreajuste, no elimina automáticamente variables. La tolerancia y el máximo de iteraciones son controles numéricos, no variantes que se probarán para escoger el mejor resultado.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from tfm.data import project_root, save_result
from tfm.logistic import MODEL_PARAMETERS, MODEL_COLUMNS, run_logistic_experiment

ROOT = project_root()
display(pd.Series(MODEL_PARAMETERS, name="Configuración fija"))
print("Predictores:", MODEL_COLUMNS)


## 2. Entrenar con separación temporal

Cada regla se ajusta quince veces, utilizando exclusivamente los ejemplos de entrenamiento de cada partición. Se respetan la purga y el embargo del 10. Los resultados de validación no intervienen en los coeficientes ni en la referencia.

La referencia ingenua siempre predice la frecuencia de éxitos de su entrenamiento: si hubo 60 etiquetas positivas de 100, asignará 0,60 a todos los ejemplos de esa validación. Es la comparación que debe mejorar un modelo que aporte información adicional.

La celda siguiente ejecuta los ajustes. **Cada ejecución añade un intento al historial científico** `outputs/experiments/logistic_trials.jsonl`, antes de entrenar, y registra el resultado o el fallo. Repetir este mismo diseño no crea una nueva configuración candidata. Si no converge, el experimento se detiene sin cambiar parámetros automáticamente.


In [ ]:
tablas, informe = run_logistic_experiment(ROOT)
predicciones = tablas["logistic_predictions"]
mensual = tablas["logistic_monthly"]
metricas = tablas["logistic_metrics"]
coeficientes = tablas["logistic_coefficients"]

display(pd.DataFrame(informe["sample_counts"]).rename(columns={
    "strategy": "Regla", "usable_examples": "Ejemplos utilizables",
    "positive": "Etiquetas positivas", "negative": "Etiquetas negativas"}))
display(pd.DataFrame(informe["diagnostics"]).rename(columns={
    "split_id": "Partición", "strategy": "Regla", "train_examples": "Entrenamiento",
    "validation_examples": "Validación", "iterations": "Iteraciones", "status": "Estado"}))
print("Estado del experimento:", informe["experiment_status"])


## 3. Cinco predicciones no son cinco meses distintos

Cada ejemplo aparece en validación en cinco particiones. Conservamos sus cinco predicciones y calculamos su **media por regla y mes**, la referencia recibe exactamente el mismo tratamiento. Así, cada ejemplo cuenta una sola vez en el resumen principal.

La media resume un conjunto de modelos de descubrimiento. No es una trayectoria CPCV ni una cartera operable, tampoco convierte sus observaciones en independientes. Si falta alguna de las cinco probabilidades, la media queda ausente, sin promediar únicamente las restantes.

En el gráfico, los meses sin ejemplo utilizable quedan como huecos: no se rellenan ni se unen a través de esos periodos. Los marcadores permiten ver también las predicciones de meses aislados.


In [ ]:
REGLA_EJEMPLO = "momentum"
MES_EJEMPLO = mensual.loc[mensual.strategy.eq(REGLA_EJEMPLO), "target_period"].iloc[0]
display(predicciones.loc[predicciones.strategy.eq(REGLA_EJEMPLO)
                        & predicciones.target_period.eq(MES_EJEMPLO)])
display(mensual.loc[mensual.strategy.eq(REGLA_EJEMPLO)
                   & mensual.target_period.eq(MES_EJEMPLO)])

calendario = pd.period_range(mensual.target_period.min(), mensual.target_period.max(), freq="M")
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, sharey=True)
for ax, regla in zip(axes, ("momentum", "reversal")):
    datos = (mensual.loc[mensual.strategy.eq(regla)].set_index("target_period")
             .reindex(calendario.astype(str)))
    fechas = calendario.to_timestamp()
    ax.plot(fechas, datos.probability_logistic, label="Logística L2", linewidth=0.8, marker=".", markersize=3)
    ax.plot(fechas, datos.probability_naive, label="Referencia ingenua", linewidth=1, marker=".", markersize=3)
    ax.set(title=regla, ylabel="Probabilidad media", ylim=(0, 1))
    ax.grid(alpha=0.25)
    ax.legend()
axes[-1].set_xlabel("Mes objetivo")
fig.suptitle("Probabilidades de validación de descubrimiento: media de cinco modelos")
fig.tight_layout()
plt.show()


## 4. Cómo juzgar las probabilidades

**Log-loss** evalúa la probabilidad asignada a la etiqueta real: cuanto menor, mejor. Equivocarse con una seguridad del 99 % recibe una penalización mayor que equivocarse con un 51 %. Las probabilidades originales se conservan, solo al calcular esta métrica se limita su valor a la precisión numérica para evitar logaritmos de cero.

**AUC** mide si los ejemplos favorables suelen recibir probabilidades mayores que los desfavorables. Un valor 0,50 corresponde a ausencia de ordenación útil como referencia, no a acertar exactamente la mitad de las operaciones. No requiere un umbral de aceptar/rechazar y no mide ganancias. La referencia ingenua tiene AUC 0,5 dentro de una partición con ambas clases, pero su AUC agregada puede variar porque su probabilidad cambia entre particiones.

La tabla principal cuenta una vez cada ejemplo. Después se muestran las métricas por partición, que están relacionadas entre sí y no constituyen quince experimentos independientes. Si solo hay una clase en validación, AUC queda ausente, log-loss sigue calculándose con las dos etiquetas posibles fijadas.


In [ ]:
principal = metricas.loc[metricas.aggregation.eq("monthly_mean")].copy()
display(principal[["strategy", "model", "examples", "positive", "negative", "log_loss", "auc", "status"]]
        .rename(columns={"strategy": "Regla", "model": "Método", "examples": "Ejemplos",
                         "log_loss": "Log-loss", "auc": "AUC", "status": "Estado"}))

comparaciones = []
for regla, grupo in principal.groupby("strategy"):
    modelo = grupo.loc[grupo.model.eq("logistic")].iloc[0]
    ingenuo = grupo.loc[grupo.model.eq("naive")].iloc[0]
    mejora = ingenuo.log_loss - modelo.log_loss
    comparaciones.append({"Regla": regla, "Mejora de log-loss (positiva es favorable)": mejora,
        "AUC al menos 0,54": bool(modelo.auc >= 0.54) if pd.notna(modelo.auc) else None,
        "Log-loss menor que el ingenuo": bool(mejora > 0) if pd.notna(mejora) else None,
        "Puerta completa": "Pendiente de sombras del modelo"})
display(pd.DataFrame(comparaciones))
display(metricas.loc[metricas.aggregation.eq("partition"),
                    ["split_id", "strategy", "model", "examples", "log_loss", "auc", "status"]])


## 5. Coeficientes: descripción del ajuste

Un coeficiente positivo aumenta la probabilidad estimada de éxito al aumentar esa característica y mantener las demás constantes. El efecto sobre la probabilidad no es constante: depende también del resto de características. Estos coeficientes no son efectos causales ni rentabilidades esperadas.

Mostramos su mediana y extremos entre particiones para describir la variación de los ajustes. Son coeficientes de características estandarizadas con historia anterior. La correlación entre variables puede hacer que el reparto de efectos cambie, esta tabla no sustituye a Clustered MDA o SHAP ni selecciona variables.


In [ ]:
resumen_coeficientes = (coeficientes.groupby(["strategy", "term"], sort=True).coefficient
                        .agg(ajustes_disponibles="count", mediana="median", minimo="min", maximo="max")
                        .reset_index())
display(resumen_coeficientes.rename(columns={"strategy": "Regla", "term": "Término"}))


## 6. Qué queda pendiente y qué guardamos

La guía pide AUC aproximadamente a partir de 0,54, log-loss mejor que el ingenuo y mejores resultados que sombras. **La puerta completa permanece pendiente**, aunque se cumplan las dos primeras condiciones. Las sombras de regresiones del 09 no son las sombras de este modelo.

Después se concretarán las sombras del clasificador, Clustered MDA con validación interna, los otros modelos y SHAP. Antes de simular el filtro habrá que fijar cómo convertir las probabilidades en decisiones. No se calculan aquí Sharpe, DSR/PBO o rentabilidades de una cartera filtrada, ni se toca el periodo desde 1990.

La celda siguiente guarda cuatro tablas con sus informes en `data/processed/`. Si falla, repítela completa antes de utilizar las salidas, no es necesario repetir los ajustes para guardar de nuevo. El historial de intentos se conserva aparte. Antes del commit, limpia las salidas del notebook y guarda.

Referencias: guía 1.B (07/11/2026), López de Prado (2018), capítulos 7 y 12, y [ejemplo de logística de Jansen](https://github.com/stefan-jansen/machine-learning-for-trading/blob/91c903e71c1d0f1b9015f18551fadbb60bcad132/11_ml_pipeline/03_logistic_classification.py), usado como apoyo de organización sin copiar código. Las métricas y el optimizador usan scikit-learn, las referencias y las decisiones específicas están en [el protocolo](../docs/primer_modelo.md).


In [ ]:
guardados = []
for nombre, tabla in tablas.items():
    salida = save_result(ROOT, nombre, tabla, informe)
    guardados.append({"Tabla": nombre + ".parquet", "Filas": salida["rows"]})
display(pd.DataFrame(guardados))
print("Historial:", informe["trial_log"])
print("Puerta de capacidad predictiva: pendiente de sombras del modelo.")
